Convert filtered 8–11mer peptides into fixed-length (11) one-hot encoded vectors (11×21 = 231 features) ready for model training.

## Filtering to 8-11mer peptides

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

In [25]:
DATA_DIR = Path("..") / "data" / "processed"
cleaned_path = DATA_DIR / "combined_cleaned.csv"

df = pd.read_csv(cleaned_path)

conflict = df.groupby("peptide")["label"].nunique()
print(f"Peptides with conflicting labels (both 0 and 1): {(conflict > 1).sum()}")

df["length"] = df["peptide"].str.len()
df = df[df["length"].between(8, 11)].copy()
print(f"Filtered to 8-11mers: {len(df):,} rows")
print(df["label"].value_counts(normalize=True))

Peptides with conflicting labels (both 0 and 1): 0
Filtered to 8-11mers: 47,602 rows
label
0    0.702807
1    0.297193
Name: proportion, dtype: float64


## Encoding strategy

Sequences are padded to length 11 (the maximum MHC class I length) using `-` as a
gap character. The one-hot encoding uses 21 columns per position: one per standard
amino acid, plus a dedicated padding column. The padding character maps to column 20
and is not treated as an amino acid by the model.

Padding is right-justified (C-terminal end) which is a reasonable default for MHC
class I peptides, where the anchor residues at positions 2 and 9 (in 9-mers) are
the most conserved. A more principled approach would be to center-pad (anchoring on
P2 and PΩ), but right-padding is used here for simplicity.


## Padding sequences to a fixed length of 11

In [ ]:
MAX_LEN = 11
AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")
aa_to_idx = {aa: i for i, aa in enumerate(AA_LIST)}

def pad_sequence(seq, max_len=MAX_LEN):
    return seq.ljust(max_len, "-")  # "-" as padding token

## One-hot encode

In [ ]:
def one_hot_encode(seq, max_len=MAX_LEN):
    padded = pad_sequence(seq, max_len)
    encoding = np.zeros((max_len, 21))  # 20 AA + 1 padding
    for i, aa in enumerate(padded):
        if aa in aa_to_idx:
            encoding[i, aa_to_idx[aa]] = 1
        else:
            encoding[i, 20] = 1  # padding column
    return encoding.flatten()

X = np.array([one_hot_encode(seq) for seq in df["peptide"]])
y = df["label"].values
print(f"Feature matrix shape: {X.shape}")  # expect (n_samples, 11*21=231)

Feature matrix shape: (47602, 231)


## Train/test split

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]:,}  Test: {X_test.shape[0]:,}")

Train: 38,081  Test: 9,521


In [26]:
np.savez(DATA_DIR / "features.npz",
         X_train=X_train, X_test=X_test,
         y_train=y_train, y_test=y_test)
print("Saved features.npz")


Saved features.npz



## Summary & next steps

- Produced fixed-length one-hot features (11×21 = 231) from filtered 8–11mer peptides and saved train/test splits.
- Preprocessing choices: right (C-terminal) padding with '-' and a dedicated padding column; sequences validated against the 20 standard AAs.
- Modeling plan: start with logistic regression and random forest as baselines, then try a simple 1D CNN or embedding-based model.
- Evaluation plan: use stratified CV or hold-out test, report AUC, ROC, confusion matrix, and class-weighting or resampling if imbalance is large.
- Possible improvements: center-padding anchored on P2/PΩ, explicit truncation guard, and persisting the encoder (json/pickle) before training.